In [3]:
# Load the patient dataset and view summary
import pandas as pd

patient_df = pd.read_csv("Patient_Data.csv")
billing_df=pd.read_csv("Billing_Data.csv")
patient_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [88]:
# Define billing-relevant columns
billing_columns = ['PatientID', 'Department', 'Doctor', 'BillAmount']

# Select only these columns from patient_df
patient_billing_df = patient_df[billing_columns].copy()

print(patient_billing_df.columns)



Index(['PatientID', 'Department', 'Doctor', 'BillAmount'], dtype='object')


In [90]:
# Define billing-relevant columns
billing_columns = ['PatientID', 'Department', 'Doctor', 'BillAmount']
patient_billing_df = patient_df[billing_columns]

# Display the billing dataframe
print(patient_billing_df.head())



   PatientID   Department     Doctor  BillAmount
0        101  Orthopedics  Dr. Kumar       15000
1        102   Cardiology    Dr. Rao       20000
2        103    Neurology  Dr. Mehta       18000
3        104  Dermatology  Dr. Singh       12000


In [44]:
# Drop Administrative columns
patient_df = patient_df.drop(
    columns=['ReceptionistID', 'CheckInTime'],
    errors='ignore'
)
# Verify the result
print(patient_df.columns)


Index(['PatientID', 'Name', 'Department', 'Doctor', 'BillAmount'], dtype='object')


In [69]:
#  Group by Department to get total bill amount

total_bill_by_department = patient_billing_df.groupby('Department')['BillAmount'].sum().reset_index()

print(total_bill_by_department)

    Department    BillAmount
0   Cardiology  11200.000000
1  Dermatology   6233.333333
2    Neurology   6233.333333
3  Orthopedics   7500.000000


In [75]:
df = pd.DataFrame({
    'PatientID': [101, 102, 103, 101, 104],
    'Department': ['Orthopedics', 'Cardiology', 'Neurology', 'Orthopedics', 'Dermatology'],
    'Doctor': ['Dr. Kumar', 'Dr. Rao', 'Dr. Mehta', 'Dr. Kumar', 'Dr. Singh']
})

# Remove exact duplicate rows
df_unique = df.drop_duplicates()

print(df_unique)


   PatientID   Department     Doctor
0        101  Orthopedics  Dr. Kumar
1        102   Cardiology    Dr. Rao
2        103    Neurology  Dr. Mehta
4        104  Dermatology  Dr. Singh


In [55]:
# Fill missing BillAmount with mean
mean_bill = patient_billing_df['BillAmount'].mean()
patient_billing_df['BillAmount'] = patient_billing_df['BillAmount'].fillna(mean_bill)

# Verify no missing values remain
print("Missing BillAmount:", patient_billing_df['BillAmount'].isna().sum())

# Display the result
print("Mean BillAmount:", mean_bill)




Missing BillAmount: 0
Mean BillAmount: 6233.333333333333


In [62]:
# Merge billing dataset with patient dataset on PatientID
merged_df = pd.merge(
    billing_df,
    patient_df,
    on='PatientID',
    how='inner'
)
# Verify the merged result
print("rows of merged DataFrame:")
print(merged_df.head())



rows of merged DataFrame:
   PatientID  InsuranceCovered  FinalAmount     Name   Department     Doctor  \
0        101              2000         3000    Alice   Cardiology  Dr. Smith   
1        102              1500         3500      Bob    Neurology   Dr. John   
2        103              2500         5000  Charlie  Orthopedics    Dr. Lee   
3        104              3000         3200    David   Cardiology  Dr. Smith   
4        105              1000         4000      Eva  Dermatology   Dr. Rose   

   BillAmount  
0      5000.0  
1         NaN  
2      7500.0  
3      6200.0  
4         NaN  


In [60]:
# New patients to add
new_patients = pd.DataFrame({
    'PatientID': [301, 302],
    'Department': ['Neurology', 'Cardiology'],
    'Doctor': ['Dr. Mehta', 'Dr. Rao'],
    'BillAmount': [25000, 18000]
})

# Concatenate row-wise
updated_df = pd.concat([merged_df, new_patients], axis=0, ignore_index=True)

# Display the updated dataframe
print(updated_df)


   PatientID   Department     Doctor  BillAmount
0        101  Orthopedics  Dr. Kumar       15000
1        102   Cardiology    Dr. Rao       20000
2        103    Neurology  Dr. Mehta       18000
3        301    Neurology  Dr. Mehta       25000
4        302   Cardiology    Dr. Rao       18000


In [87]:
# Concatenate new billing columns (column-wise)
billing_cols = pd.DataFrame({
    'InsuranceCovered': [True] * len(updated_df),
    'FinalAmount': updated_df['BillAmount'] * 0.9
})

final_dataset = pd.concat(
    [updated_df, billing_cols],
    axis=1
)
# Verify the final dataset
print(" rows of final dataset:")
print(final_dataset.head())


 rows of final dataset:
   PatientID   Department     Doctor  BillAmount  InsuranceCovered  \
0        101  Orthopedics  Dr. Kumar       15000              True   
1        102   Cardiology    Dr. Rao       20000              True   
2        103    Neurology  Dr. Mehta       18000              True   
3        301    Neurology  Dr. Mehta       25000              True   
4        302   Cardiology    Dr. Rao       18000              True   

   FinalAmount  
0      13500.0  
1      18000.0  
2      16200.0  
3      22500.0  
4      16200.0  
